In [4]:
!pip install requests
!pip install json
!pip install pandas


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: Could not find a version that satisfies the requirement json (from versions: none)

[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: No matching distribution found for json



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
import requests
import pandas as pd
import time

pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)

TOTW_URL = "https://www.fotmob.com/api/data/team-of-the-week/team"

ligas_totw = {
    "Premier League": {"id": 47, "rondas": 38},
    "LaLiga":         {"id": 87, "rondas": 38},
    "Serie A":        {"id": 55, "rondas": 38},
    "Bundesliga":     {"id": 54, "rondas": 34},
    "Ligue 1":        {"id": 53, "rondas": 34},
}


def obtener_totw_jornada(id_liga, round_id, season="2025/2026", mapa_equipos=None):
    """Descarga el Equipo de la Semana de UNA jornada concreta."""
    params = {"leagueId": id_liga, "roundId": round_id, "season": season}
    r = requests.get(TOTW_URL, params=params)
    if r.status_code != 200:
        return None

    data = r.json()
    if not isinstance(data, list) or not data:
        return None

    filas = []
    for jugador in data:
        nombre = jugador.get("name", {}) or {}
        rating = jugador.get("rating", {}) or {}
        layout = jugador.get("verticalLayout", {}) or {}
        team_id = jugador.get("teamId")

        filas.append({
            "player_id": jugador.get("id"),
            "player": nombre.get("fullName"),
            "rating": rating.get("num"),
            "teamId": team_id,
            "team": (mapa_equipos or {}).get(team_id),
            "pos_x": layout.get("x"),
            "pos_y": layout.get("y"),
            "matchId": jugador.get("matchId"),
            "isTots": jugador.get("isTots"),
            "Jornada": round_id,
        })

    df = pd.DataFrame(filas)
    df["Temporada"] = season
    return df


def mapa_equipos_liga(id_liga, season="2025/2026", pais="ESP"):
    """
    Usa la tabla de clasificación (que ya sabemos leer bien) para
    construir un diccionario {teamId: nombre_equipo}.
    """
    params = {"id": id_liga, "ccode3": pais, "season": season}
    r = requests.get("https://www.fotmob.com/api/data/leagues", params=params)
    r.raise_for_status()
    data = r.json()

    tablas = data.get("table", [])
    for bloque in tablas:
        tabla_obj = bloque.get("data", {}).get("table", {})
        filas_all = tabla_obj.get("all")
        if filas_all:
            return {f["id"]: f["name"] for f in filas_all if "id" in f and "name" in f}
    return {}


def obtener_totw_temporada(nombre_liga, id_liga, num_rondas, season="2025/2026", pais="ESP", pausa=0.3, debug=False):
    """
    Descarga el Equipo de la Semana de TODAS las jornadas de la temporada
    para una liga, y lo devuelve como un único DataFrame.
    """
    mapa_equipos = mapa_equipos_liga(id_liga, season, pais)

    dfs = []
    for ronda in range(1, num_rondas + 1):
        df_ronda = obtener_totw_jornada(id_liga, ronda, season, mapa_equipos)
        if df_ronda is not None:
            df_ronda["Liga"] = nombre_liga
            dfs.append(df_ronda)
            if debug:
                print(f"  ✔ {nombre_liga} jornada {ronda}: {len(df_ronda)} jugadores")
        else:
            if debug:
                print(f"  — {nombre_liga} jornada {ronda}: sin datos (aún no jugada o no disponible)")
        time.sleep(pausa)  # para no machacar la API con requests seguidos

    if not dfs:
        return pd.DataFrame()

    return pd.concat(dfs, ignore_index=True)


def totw_global_temporada(df_totw_temporada):
    """
    A partir del DataFrame de todas las jornadas de una liga, calcula
    el 'global' de temporada: cuántas veces ha entrado cada jugador
    en el Equipo de la Semana, con su rating medio.
    """
    if df_totw_temporada.empty:
        return pd.DataFrame()

    resumen = (
        df_totw_temporada
        .assign(rating=pd.to_numeric(df_totw_temporada["rating"], errors="coerce"))
        .groupby(["player_id", "player", "team", "Liga"], as_index=False)
        .agg(veces_en_totw=("Jornada", "count"), rating_medio=("rating", "mean"))
        .sort_values(["veces_en_totw", "rating_medio"], ascending=[False, False])
        .reset_index(drop=True)
    )
    return resumen

Ejecutarlo para las 5 grandes ligas

In [6]:
season_in, season_out = 2025, 2026
temporada = f"{season_in}/{season_out}"

paises_liga = {
    "Premier League": "ENG",
    "LaLiga": "ESP",
    "Serie A": "ITA",
    "Bundesliga": "GER",
    "Ligue 1": "FRA",
}

totw_por_liga = {}
totw_global_por_liga = {}

for nombre_liga, info in ligas_totw.items():
    print(f"\n🏆 {nombre_liga}")
    df_temp = obtener_totw_temporada(
        nombre_liga, info["id"], info["rondas"],
        season=temporada, pais=paises_liga[nombre_liga], debug=True
    )
    totw_por_liga[nombre_liga] = df_temp
    totw_global_por_liga[nombre_liga] = totw_global_temporada(df_temp)


🏆 Premier League
  ✔ Premier League jornada 1: 11 jugadores
  ✔ Premier League jornada 2: 11 jugadores
  ✔ Premier League jornada 3: 11 jugadores
  ✔ Premier League jornada 4: 11 jugadores
  ✔ Premier League jornada 5: 11 jugadores
  ✔ Premier League jornada 6: 11 jugadores
  ✔ Premier League jornada 7: 11 jugadores
  ✔ Premier League jornada 8: 11 jugadores
  ✔ Premier League jornada 9: 11 jugadores
  ✔ Premier League jornada 10: 11 jugadores
  ✔ Premier League jornada 11: 11 jugadores
  ✔ Premier League jornada 12: 11 jugadores
  ✔ Premier League jornada 13: 11 jugadores
  ✔ Premier League jornada 14: 11 jugadores
  ✔ Premier League jornada 15: 11 jugadores
  ✔ Premier League jornada 16: 11 jugadores
  ✔ Premier League jornada 17: 11 jugadores
  ✔ Premier League jornada 18: 11 jugadores
  ✔ Premier League jornada 19: 11 jugadores
  ✔ Premier League jornada 20: 11 jugadores
  ✔ Premier League jornada 21: 11 jugadores
  ✔ Premier League jornada 22: 11 jugadores
  ✔ Premier League jorn

Acceder a los resultados

In [7]:
# TOTW de una jornada concreta de una liga
totw_por_liga["LaLiga"][totw_por_liga["LaLiga"]["Jornada"] == 37]

# Ranking global de temporada (jugadores con más apariciones en el TOTW)
totw_global_por_liga["LaLiga"].head(15)

# Todas las ligas juntas, global de temporada
df_global_5_ligas = pd.concat(totw_global_por_liga.values(), ignore_index=True)
df_global_5_ligas.sort_values("veces_en_totw", ascending=False).head(20)

,player_id,player,team,Liga,veces_en_totw,rating_medio
0,422685,Bruno Fernandes,Manchester United,Premier League,14,8.692857
634,194165,Harry Kane,Bayern München,Bundesliga,13,9.192308
214,1467236,Lamine Yamal,Barcelona,LaLiga,11,9.100000
635,1029063,Michael Olise,Bayern München,Bundesliga,10,8.850000
430,1347574,Nico Paz,Como,Serie A,10,8.650000
1,737066,Erling Haaland,Manchester City,Premier League,10,9.020000
636,806669,David Raum,RB Leipzig,Bundesliga,9,8.477778
217,1083323,Pedri,Barcelona,LaLiga,9,8.455556
431,605224,Federico Dimarco,Inter,Serie A,9,8.844444
216,517052,Vedat Muriqi,Mallorca,LaLiga,9,8.700000
